In [9]:
url = "https://raw.githubusercontent.com/Diegoalvarezca/Proyecto_Investigacion_IA_salud/refs/heads/Generaci%C3%B3n_Dataset/df_maestro_imputado_final.csv"

!wget {url} -O df_maestro_imputado_final.csv
print("Dataset descargado correctamente.")

--2026-06-16 22:21:56--  https://raw.githubusercontent.com/Diegoalvarezca/Proyecto_Investigacion_IA_salud/refs/heads/Generaci%C3%B3n_Dataset/df_maestro_imputado_final.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 7130724 (6.8M) [text/plain]
Saving to: ‘df_maestro_imputado_final.csv’

df_maestro_imputado 100%[===================>]   6.80M  --.-KB/s    in 0.08s   

2026-06-16 22:21:56 (85.5 MB/s) - ‘df_maestro_imputado_final.csv’ saved [7130724/7130724]

Dataset descargado correctamente.


In [10]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

df = pd.read_csv("df_maestro_imputado_final.csv")

# Crear la variable combinada Estado vital + Ciclo
# Esto asegura que la tasa de mortalidad y la proporción de años sea igual en Train, Val y Test
df['strata'] = df['MORTSTAT'].astype(str) + "_" + df['CICLO_ORIGEN'].astype(str)

# Primer split: 80% Train+Val, 20% Test
df_train_val, df_test = train_test_split(
    df, test_size=0.20, stratify=df['strata'], random_state=42
)

# Recalcular estratos para el segundo split
df_train_val['strata'] = df_train_val['MORTSTAT'].astype(str) + "_" + df_train_val['CICLO_ORIGEN'].astype(str)

# Segundo split: 75% Train, 25% Val (del 80% inicial, resultando en 60% Train, 20% Val, 20% Test)
df_train, df_val = train_test_split(
    df_train_val, test_size=0.25, stratify=df_train_val['strata'], random_state=42
)

print(f"Tamaño de Entrenamiento: {df_train.shape[0]} pacientes")
print(f"Tamaño de Validación: {df_val.shape[0]} pacientes")
print(f"Tamaño de Prueba: {df_test.shape[0]} pacientes")

Tamaño de Entrenamiento: 19344 pacientes
Tamaño de Validación: 6449 pacientes
Tamaño de Prueba: 6449 pacientes


In [15]:
# Contar cuántos pacientes realmente fallecieron en el set de Entrenamiento
total_muertes_train = df_train[df_train['MORTSTAT'] == 1].shape[0]

print(f"Total de eventos (muertes) en Train: {total_muertes_train}")

# Probar diferentes tamaños de K
for k in [10, 11, 12, 15, 20]:
    eventos_por_bin = total_muertes_train // k
    print(f"Si K={k} -> Cada intervalo tendrá aprox. {eventos_por_bin} muertes")


Total de eventos (muertes) en Train: 1761
Si K=10 -> Cada intervalo tendrá aprox. 176 muertes
Si K=11 -> Cada intervalo tendrá aprox. 160 muertes
Si K=12 -> Cada intervalo tendrá aprox. 146 muertes
Si K=15 -> Cada intervalo tendrá aprox. 117 muertes
Si K=20 -> Cada intervalo tendrá aprox. 88 muertes


In [17]:
# --------------------------------------------------
# DISCRETIZACIÓN DEL TARGET CORREGIDA
# --------------------------------------------------

# Asegurar que la columna sea numérica (errores a NaN)
df_train['PERMTH_INT'] = pd.to_numeric(df_train['PERMTH_INT'], errors='coerce')
df_val['PERMTH_INT'] = pd.to_numeric(df_val['PERMTH_INT'], errors='coerce')
df_test['PERMTH_INT'] = pd.to_numeric(df_test['PERMTH_INT'], errors='coerce')

# Opcional pero recomendado para recuperar el min/max general real:
tiempo_min_global = min(df_train['PERMTH_INT'].min(), df_val['PERMTH_INT'].min(), df_test['PERMTH_INT'].min())
tiempo_max_global = max(df_train['PERMTH_INT'].max(), df_val['PERMTH_INT'].max(), df_test['PERMTH_INT'].max())

num_time_bins = 15

# Calcular cuantiles
tiempos_evento = df_train[df_train['MORTSTAT'] == 1]['PERMTH_INT']
_, bins_temporales = pd.qcut(tiempos_evento, q=num_time_bins, retbins=True, labels=False, duplicates='drop')

# Ajustar los extremos para cubrir el dataset completo
bins_temporales[0] = tiempo_min_global - 1
bins_temporales[-1] = tiempo_max_global + 1

def discretizar_target(dataframe, bins):
    # Asignar cada tiempo de seguimiento a su respectivo intervalo
    tiempo_discreto = pd.cut(dataframe['PERMTH_INT'], bins=bins, labels=False, include_lowest=True)
    estado = dataframe['MORTSTAT'].values
    return tiempo_discreto.values, estado

# Extraer los targets transformados
y_train_time, y_train_event = discretizar_target(df_train, bins_temporales)
y_val_time, y_val_event = discretizar_target(df_val, bins_temporales)
y_test_time, y_test_event = discretizar_target(df_test, bins_temporales)

# La cantidad final de intervalos que el modelo usará (puede ser menor a 15 si se fusionaron empates)
num_bins_finales = len(bins_temporales) - 1

print(f"Discretización completa")
print(f"El modelo tendrá exactamente {num_bins_finales} neuronas de salida en el tiempo.")


Discretización completa
El modelo tendrá exactamente 3 neuronas de salida en el tiempo.


In [18]:
import pandas as pd

# Ver las estadísticas reales del tiempo de seguimiento de los pacientes fallecidos
tiempos_fallecidos = df_train[df_train['MORTSTAT'] == 1]['PERMTH_INT']

print("=== RADIOGRAFÍA DEL TIEMPO DE SUPERVIVENCIA ===")
print(tiempos_fallecidos.describe())
print("\nLos 10 valores de tiempo más comunes y cuántos pacientes los tienen:")
print(tiempos_fallecidos.value_counts().head(10))

=== RADIOGRAFÍA DEL TIEMPO DE SUPERVIVENCIA ===
count    1761.000000
mean        1.240772
std         3.221370
min         0.000000
25%         0.000000
50%         0.000000
75%         0.000000
max        11.000000
Name: PERMTH_INT, dtype: float64

Los 10 valores de tiempo más comunes y cuántos pacientes los tienen:
PERMTH_INT
0.0     1375
1.0      195
10.0     111
11.0      80
Name: count, dtype: int64
